<div style="
    background: linear-gradient(135deg, #0F172A 0%, #111827 100%);
    padding: 35px;
    border-radius: 22px;
    border-bottom: 8px solid #06B6D4;
    box-shadow: 0 10px 30px rgba(15,23,42,0.35);
    font-family: Segoe UI, Arial, sans-serif;
">

<h1 style="
    color: white;
    margin: 0;
    font-size: 42px;
    font-weight: 800;
    letter-spacing: 1px;
">
NEXORA ANALYTICS
</h1>

<p style="
    color: #CBD5E1;
    font-size: 20px;
    margin-top: 12px;
    margin-bottom: 0;
">
Projet Global — Data Science, Détection de Fraude,
Segmentation Client et MLOps
</p>

</div>

<div style="
    background:#F8FAFC;
    padding:28px;
    border-radius:18px;
    border-left:10px solid #06B6D4;
    box-shadow:0 6px 18px rgba(15,23,42,0.10);
    font-family:Segoe UI, Arial, sans-serif;
    margin-top:20px;
">

<h1 style="
    color:#0F172A;
    margin:0;
    font-size:32px;
    font-weight:800;
">
Exercice 2 — Segmentation intelligente des clients avec Clustering
</h1>

<p style="
    color:#475569;
    font-size:17px;
    margin-top:12px;
">
Identification automatique des profils clients grâce aux techniques avancées de clustering.
</p>

</div>

<div style="
    border-left:6px solid #06B6D4;
    background:#ECFEFF;
    padding:14px 18px;
    border-radius:12px;
    font-family:Segoe UI;
    margin-top:20px;
">

<h3 style="
    margin:0;
    color:#0F172A;
">
3. Clustering
</h3>

</div>

In [1]:
# ============================================
# IMPORTATION DES LIBRAIRIES
# ============================================

import pandas as pd
import numpy as np

from sklearn.cluster import (
    KMeans,
    DBSCAN,
    AgglomerativeClustering
)

from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score
)

import plotly.express as px

print("Librairies importées avec succès")

Librairies importées avec succès


In [2]:
# ============================================
# CHARGEMENT DES DONNÉES
# ============================================

scaled_df = pd.read_csv(
    "../data/processed/cluster_scaled.csv"
)

pca_df = pd.read_csv(
    "../data/processed/cluster_pca.csv"
)

print("Données chargées")

print(scaled_df.shape)
print(pca_df.shape)

Données chargées
(2240, 29)
(2240, 2)


<div style="
    display:inline-block;
    border-left:5px solid #06B6D4;
    background:#ECFEFF;
    padding:10px 14px;
    border-radius:10px;
    font-family:Segoe UI;
    margin-top:15px;
    margin-bottom:10px;
    box-shadow:0 2px 8px rgba(15,23,42,0.06);
">

<h3 style="
    margin:0;
    color:#0F172A;
    font-size:18px;
    font-weight:600;
">
. ELBOW METHOD — KMEANS
</h3>

</div>

In [3]:
# ============================================
# ELBOW METHOD
# ============================================

inertia = []

K = range(2, 11)

for k in K:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(scaled_df)

    inertia.append(kmeans.inertia_)

elbow_df = pd.DataFrame({
    "Clusters": list(K),
    "Inertia": inertia
})

fig = px.line(
    elbow_df,
    x="Clusters",
    y="Inertia",
    markers=True,
    title="Elbow Method pour KMeans"
)

fig.update_layout(
    template="plotly_white",
    height=600,
    title_x=0.5
)

fig.show()

<div style="
    background:#F8FAFC;
    border-left:6px solid #7C3AED;
    padding:18px 20px;
    border-radius:12px;
    margin-top:15px;
    font-family:Segoe UI;
">

<h3 style="margin-top:0; color:#0F172A;">
Choix du nombre de clusters
</h3>

<p style="color:#334155; line-height:1.7;">

L’analyse de la méthode du coude (Elbow Method) montre un point de cassure autour de 5 clusters.

Avant cette valeur, l’ajout de nouveaux clusters réduit fortement l’inertie.

Au-delà de 5 clusters, les améliorations deviennent plus faibles et moins significatives.

Le choix de 5 segments permet donc d’obtenir :
</p>

<ul style="color:#334155; line-height:1.7;">
<li>une segmentation suffisamment détaillée ;</li>
<li>une bonne séparation des profils clients ;</li>
<li>un équilibre entre performance et interprétabilité.</li>
</ul>

</div>

<div style="
    background:#F8FAFC;
    border-left:6px solid #7C3AED;
    padding:18px 20px;
    border-radius:12px;
    margin-top:15px;
    font-family:Segoe UI;
">

<h3 style="margin-top:0; color:#0F172A;">
Interprétation métier
</h3>

<p style="color:#334155; line-height:1.7;">
L’Elbow Method permet d’identifier le nombre optimal de segments clients.
</p>

<p style="color:#334155; line-height:1.7;">
Le point de cassure de la courbe indique généralement le nombre de clusters offrant le meilleur équilibre entre :
</p>

<ul style="color:#334155; line-height:1.7;">
<li>précision ;</li>
<li>simplicité ;</li>
<li>qualité des regroupements.</li>
</ul>

</div>

<div style="
    display:inline-block;
    border-left:5px solid #06B6D4;
    background:#ECFEFF;
    padding:10px 14px;
    border-radius:10px;
    font-family:Segoe UI;
    margin-top:15px;
    margin-bottom:10px;
    box-shadow:0 2px 8px rgba(15,23,42,0.06);
">

<h3 style="
    margin:0;
    color:#0F172A;
    font-size:18px;
    font-weight:600;
">
 •	KMEANS
</h3>

</div>

In [4]:
# ============================================
# KMEANS
# ============================================

kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

kmeans_labels = kmeans.fit_predict(
    scaled_df
)

pca_df["KMeans_Cluster"] = kmeans_labels

print("KMeans terminé")

KMeans terminé


In [5]:
# ============================================
# VISUALISATION KMEANS
# ============================================

fig = px.scatter(
    pca_df,

    x="PCA1",
    y="PCA2",

    color=pca_df["KMeans_Cluster"].astype(str),

    title="Segmentation clients — KMeans",

    color_discrete_sequence=[
        "#06B6D4",
        "#7C3AED",
        "#0F172A",
        "#94A3B8"
    ]
)

fig.update_layout(
    template="plotly_white",
    height=700,
    title_x=0.5
)

fig.show()

<div style="
    display:inline-block;
    border-left:5px solid #06B6D4;
    background:#ECFEFF;
    padding:10px 14px;
    border-radius:10px;
    font-family:Segoe UI;
    margin-top:15px;
    margin-bottom:10px;
    box-shadow:0 2px 8px rgba(15,23,42,0.06);
">

<h3 style="
    margin:0;
    color:#0F172A;
    font-size:18px;
    font-weight:600;
">
 •	DBSCAN
</h3>

</div>

In [6]:
# ============================================
# DBSCAN
# ============================================

dbscan = DBSCAN(
    eps=2,
    min_samples=10
)

dbscan_labels = dbscan.fit_predict(
    scaled_df
)

pca_df["DBSCAN_Cluster"] = dbscan_labels

print("DBSCAN terminé")

DBSCAN terminé


In [7]:
# ============================================
# VISUALISATION DBSCAN
# ============================================

fig = px.scatter(
    pca_df,

    x="PCA1",
    y="PCA2",

    color=pca_df["DBSCAN_Cluster"].astype(str),

    title="Segmentation clients — DBSCAN"
)

fig.update_layout(
    template="plotly_white",
    height=700,
    title_x=0.5
)

fig.show()

<div style="
    display:inline-block;
    border-left:5px solid #06B6D4;
    background:#ECFEFF;
    padding:10px 14px;
    border-radius:10px;
    font-family:Segoe UI;
    margin-top:15px;
    margin-bottom:10px;
    box-shadow:0 2px 8px rgba(15,23,42,0.06);
">

<h3 style="
    margin:0;
    color:#0F172A;
    font-size:18px;
    font-weight:600;
">
 •	AGGLOMERATIVE CLUSTERING
</h3>

</div>

In [12]:
# ============================================
# AGGLOMERATIVE CLUSTERING
# ============================================

agglo = AgglomerativeClustering(
    n_clusters=5
)

agglo_labels = agglo.fit_predict(
    scaled_df
)

pca_df["Agglo_Cluster"] = agglo_labels

print("Agglomerative terminé")

Agglomerative terminé


In [13]:
# ============================================
# VISUALISATION AGGLOMERATIVE
# ============================================

fig = px.scatter(
    pca_df,

    x="PCA1",
    y="PCA2",

    color=pca_df["Agglo_Cluster"].astype(str),

    title="Segmentation clients — Agglomerative Clustering"
)

fig.update_layout(
    template="plotly_white",
    height=700,
    title_x=0.5
)

fig.show()

<div style="
    display:inline-block;
    border-left:5px solid #06B6D4;
    background:#ECFEFF;
    padding:10px 14px;
    border-radius:10px;
    font-family:Segoe UI;
    margin-top:15px;
    margin-bottom:10px;
    box-shadow:0 2px 8px rgba(15,23,42,0.06);
">

<h3 style="
    margin:0;
    color:#0F172A;
    font-size:18px;
    font-weight:600;
">
 •	GAUSSIAN MIXTURE MODELS
</h3>

</div>

In [14]:
# ============================================
# GAUSSIAN MIXTURE MODEL
# ============================================

gmm = GaussianMixture(
    n_components=5,
    random_state=42
)

gmm_labels = gmm.fit_predict(
    scaled_df
)

pca_df["GMM_Cluster"] = gmm_labels

print("GMM terminé")

GMM terminé


In [15]:
# ============================================
# VISUALISATION GMM
# ============================================

fig = px.scatter(
    pca_df,

    x="PCA1",
    y="PCA2",

    color=pca_df["GMM_Cluster"].astype(str),

    title="Segmentation clients — Gaussian Mixture Model"
)

fig.update_layout(
    template="plotly_white",
    height=700,
    title_x=0.5
)

fig.show()

<div style="
    border-left:6px solid #06B6D4;
    background:#ECFEFF;
    padding:14px 18px;
    border-radius:12px;
    font-family:Segoe UI;
    margin-top:20px;
">

<h3 style="
    margin:0;
    color:#0F172A;
">
4. Évaluation des clusters
</h3

</div>

In [16]:
# ============================================
# ÉVALUATION DES CLUSTERS
# ============================================

evaluation_results = []

models = {
    "KMeans": kmeans_labels,
    "DBSCAN": dbscan_labels,
    "Agglomerative": agglo_labels,
    "GMM": gmm_labels
}

for name, labels in models.items():

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    if n_clusters > 1:
        silhouette = silhouette_score(scaled_df, labels)
        davies = davies_bouldin_score(scaled_df, labels)
    else:
        silhouette = np.nan
        davies = np.nan

    evaluation_results.append({
        "Modèle": name,
        "Nombre de clusters": n_clusters,
        "Silhouette Score": silhouette,
        "Davies-Bouldin Score": davies
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,Modèle,Nombre de clusters,Silhouette Score,Davies-Bouldin Score
0,KMeans,5,0.158633,1.985033
1,DBSCAN,3,-0.148812,2.196273
2,Agglomerative,5,0.138433,1.884384
3,GMM,5,0.108350,2.688705


In [17]:
# ============================================
# VISUALISATION DES SCORES
# ============================================

fig = px.bar(
    evaluation_df,

    x="Modèle",
    y="Silhouette Score",

    color="Silhouette Score",

    text="Silhouette Score",

    color_continuous_scale=[
        "#06B6D4",
        "#7C3AED",
        "#0F172A"
    ],

    title="Comparaison des modèles de clustering"
)

fig.update_traces(
    texttemplate='%{text:.3f}',
    textposition='outside'
)

fig.update_layout(
    template="plotly_white",
    height=650,
    title_x=0.5,
    coloraxis_showscale=False
)

fig.show()

<div style="
    background:#F8FAFC;
    border-left:6px solid #7C3AED;
    padding:18px 20px;
    border-radius:12px;
    margin-top:15px;
    font-family:Segoe UI;
">

<h3 style="margin-top:0; color:#0F172A;">
Interprétation métier
</h3>

<p style="color:#334155; line-height:1.7;">
Les métriques d’évaluation permettent de mesurer la qualité des segments clients générés par les algorithmes.
</p>

<ul style="color:#334155; line-height:1.7;">
<li>un Silhouette Score élevé indique des clusters bien séparés ;</li>
<li>un Davies-Bouldin Score faible indique des groupes homogènes ;</li>
<li>la comparaison des modèles permet d’identifier l’algorithme le plus performant.</li>
</ul>

</div>

In [18]:
# ============================================
# SAUVEGARDE DES RÉSULTATS
# ============================================

pca_df.to_csv(
    "../data/processed/client_clusters.csv",
    index=False
)

evaluation_df.to_csv(
    "../reports/clustering_scores.csv",
    index=False
)

print("Résultats sauvegardés")

Résultats sauvegardés
